# Proyecto Final — Etapa 2: Clasificación de Textos por Década (Mark 2)

**Mejoras sobre Mark 1:**
- **Modelo:** `PlanTL-GOB-ES/roberta-large-bne` — RoBERTa-Large preentrenado sobre +200 GB de texto español incluyendo documentos históricos de la BNE. Licencia Apache 2.0. Fuente: https://huggingface.co/PlanTL-GOB-ES/roberta-large-bne
- **Mixed precision (fp16):** entrena ~2× más rápido y reduce consumo de memoria.
- **Layer-wise LR Decay (LLRD):** capas inferiores reciben menor tasa de aprendizaje para preservar características preentrenadas.
- **Label smoothing (0.1):** regularización sobre la distribución objetivo.
- **Early stopping (patience=3):** evita sobreajuste.
- **Sliding window en inferencia:** para textos >MAX_LEN se promedian los logits de ventanas solapadas en vez de truncar.

## 1. Instalación y librerías

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.optim import AdamW

## 2. Configuración

In [ ]:
IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = 'COLAB_GPU' in os.environ

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/machine-learning-project"
    OUT_DIR  = "/kaggle/working"
elif IS_COLAB:
    DATA_DIR = "/content/data"
    OUT_DIR  = "/content"
else:
    DATA_DIR = "../../data"
    OUT_DIR  = "../submissions"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
EVAL_PATH  = os.path.join(DATA_DIR, "eval.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# roberta-large-bne: modelo grande entrenado en texto español histórico
# Si hay OOM reducir BATCH_SIZE a 4 o cambiar a roberta-base-bne
MODEL_NAME  = "PlanTL-GOB-ES/roberta-large-bne"
MAX_LEN     = 256
STRIDE      = 128   # solapamiento en sliding window
BATCH_SIZE  = 8
GRAD_ACCUM  = 4     # batch efectivo = 32
EPOCHS      = 7
LR          = 1e-5
LLRD        = 0.9   # decay por capa para LLRD
LABEL_SMOOTH= 0.1
PATIENCE    = 3
WARMUP_FRAC = 0.06
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP     = torch.cuda.is_available()

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  |  Mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 3. Datos y limpieza

**Limpieza mínima:** normalizar espacios y caracteres de control sin eliminar rasgos ortográficos históricos que son informativos para la tarea.

In [ ]:
def clean(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', str(text))
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)

train_df["text"] = train_df["text"].map(clean)
eval_df["text"]  = eval_df["text"].map(clean)

print(f"Train: {train_df.shape}  |  Eval: {eval_df.shape}")
train_df.head(3)

## 4. Preprocesamiento

In [ ]:
le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["decade"])
NUM_CLASSES = len(le.classes_)
print(f"Clases: {NUM_CLASSES}")

train_data, val_data = train_test_split(
    train_df, test_size=0.1, random_state=SEED, stratify=train_df["label"]
)
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

## 5. Tokenización y Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DecadeDataset(Dataset):
    """Dataset estándar para entrenamiento y validación (trunca a MAX_LEN)."""

    def __init__(self, texts, labels=None):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item


train_loader = DataLoader(
    DecadeDataset(train_data["text"], train_data["label"]),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    DecadeDataset(val_data["text"], val_data["label"]),
    batch_size=BATCH_SIZE * 2, num_workers=2, pin_memory=True,
)
print("DataLoaders listos")

## 6. Modelo y optimizador

**LLRD:** las capas inferiores del transformer reciben `LR × LLRD^(N−i)`, preservando las representaciones preentrenadas. La cabeza de clasificación recibe el LR completo.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_CLASSES
).to(DEVICE)

print(f"Parámetros: {sum(p.numel() for p in model.parameters()):,}")


def build_optimizer(model, lr, llrd):
    """AdamW con Layer-wise LR Decay y separación de weight decay."""
    no_decay = {"bias", "LayerNorm.weight"}

    # Agrupar capas del encoder por índice
    enc_layers = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "encoder.layer." in name:
            idx = int(name.split("encoder.layer.")[1].split(".")[0])
            enc_layers.setdefault(idx, {"wd": [], "no_wd": []})
            key = "no_wd" if any(nd in name for nd in no_decay) else "wd"
            enc_layers[idx][key].append(param)

    num_enc = len(enc_layers)
    groups  = []

    # Cabeza de clasificación: LR completo
    head_wd, head_no = [], []
    for name, param in model.named_parameters():
        if "classifier" in name or "pooler" in name:
            (head_no if any(nd in name for nd in no_decay) else head_wd).append(param)
    groups += [{"params": head_wd, "lr": lr, "weight_decay": 0.01},
               {"params": head_no, "lr": lr, "weight_decay": 0.0}]

    # Capas del encoder: LR decayente desde la cima hacia abajo
    for i in sorted(enc_layers.keys(), reverse=True):
        layer_lr = lr * (llrd ** (num_enc - 1 - i))
        groups += [{"params": enc_layers[i]["wd"],    "lr": layer_lr, "weight_decay": 0.01},
                   {"params": enc_layers[i]["no_wd"], "lr": layer_lr, "weight_decay": 0.0}]

    # Embeddings: LR más bajo
    emb_lr  = lr * (llrd ** num_enc)
    emb_wd, emb_no = [], []
    for name, param in model.named_parameters():
        if "embeddings" in name:
            (emb_no if any(nd in name for nd in no_decay) else emb_wd).append(param)
    groups += [{"params": emb_wd, "lr": emb_lr, "weight_decay": 0.01},
               {"params": emb_no, "lr": emb_lr, "weight_decay": 0.0}]

    return AdamW(groups)


optimizer   = build_optimizer(model, LR, LLRD)
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Total steps: {total_steps}  |  Warmup: {int(WARMUP_FRAC*total_steps)}")

## 7. Entrenamiento

**Label smoothing:** reemplaza la distribución one-hot por `(1−ε)·δ_y + ε/(C−1)` sobre las clases falsas, donde ε=0.1 y C=39.

In [ ]:
def smooth_loss(logits, labels, num_classes, smoothing):
    log_p = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        tgt = torch.full_like(log_p, smoothing / (num_classes - 1))
        tgt.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
    return -(tgt * log_p).sum(-1).mean()


def train_epoch(loader):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        lbls = batch["labels"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = smooth_loss(logits, lbls, NUM_CLASSES, LABEL_SMOOTH) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * GRAD_ACCUM
        correct    += (logits.argmax(-1) == lbls).sum().item()
        n          += len(lbls)
    return total_loss / len(loader), correct / n


def eval_epoch(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            ids    = batch["input_ids"].to(DEVICE)
            mask   = batch["attention_mask"].to(DEVICE)
            lbls   = batch["labels"].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(input_ids=ids, attention_mask=mask).logits
                loss   = smooth_loss(logits, lbls, NUM_CLASSES, LABEL_SMOOTH)
            total_loss += loss.item()
            correct    += (logits.argmax(-1) == lbls).sum().item()
            n          += len(lbls)
    return total_loss / len(loader), correct / n


best_val_acc   = 0.0
patience_count = 0
best_ckpt      = os.path.join(OUT_DIR, "best_bne_large.pt")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(train_loader)
    va_loss, va_acc = eval_epoch(val_loader)
    print(
        f"Epoch {epoch}/{EPOCHS}  "
        f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f}  "
        f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}"
    )
    if va_acc > best_val_acc:
        best_val_acc   = va_acc
        patience_count = 0
        torch.save(model.state_dict(), best_ckpt)
        print(f"  → Checkpoint guardado (val_acc={va_acc:.4f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping en epoch {epoch}")
            break

print(f"\nMejor val_acc: {best_val_acc:.4f}")

## 8. Inferencia con Sliding Window y Envío

Para textos que superan `MAX_LEN` tokens se generan **ventanas solapadas** con stride=128. Se promedian los logits de todas las ventanas antes del argmax. El 13% de textos del conjunto de evaluación se beneficia de esto.

In [ ]:
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()


@torch.no_grad()
def predict_sliding(texts):
    """Predice con ventanas solapadas para textos largos."""
    all_preds = []
    for text in texts:
        enc = tokenizer(
            str(text),
            max_length=MAX_LEN,
            stride=STRIDE,
            truncation=True,
            padding="max_length",
            return_overflowing_tokens=True,
            return_tensors="pt",
        )
        ids  = enc["input_ids"].to(DEVICE)
        mask = enc["attention_mask"].to(DEVICE)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(input_ids=ids, attention_mask=mask).logits
        # Promediar logits de todas las ventanas
        avg_logits = logits.mean(0)
        all_preds.append(avg_logits.argmax(-1).item())
    return np.array(all_preds)


print("Generando predicciones con sliding window...")
preds = predict_sliding(eval_df["text"].tolist())

submission = pd.DataFrame({
    "id":     eval_df["id"],
    "answer": le.inverse_transform(preds),
})

sub_path = os.path.join(OUT_DIR, "submission_bne_large_mark2.csv")
submission.to_csv(sub_path, index=False)
print(f"Submission guardado: {sub_path}")
print(f"\nDistribución predicciones:\n{submission['answer'].value_counts().sort_index()}")
submission.head(10)